
# TEL351 - Tarea 1: Agentes periodistas y generación automatizada de noticias

Nombre: Pedro Arce.

## Objetivo
En esta tarea desarrollarás un sistema que scrapea noticias de un sitio web público, crea agentes automatizados que generan nuevas versiones de estas noticias, y luego presenta el resultado en un sitio web HTML.

## Detalle

- Recolectar al menos 100 noticias de un sitio web de libre acceso utilizando web scraping.

- Procesar los datos obtenidos y almacenarlos de manera estructurada.

- Diseñar 5 agentes, cada uno con una personalidad distinta, que generen noticias basadas en las originales o en su interpretación.

- Construir un sitio web HTML estático que simule ser un portal de noticias con las publicaciones generadas por los agentes.

## Parte 1: Web Scraping de noticias

In [1]:

# Importar librerías necesarias
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time



### Instrucciones:
- Elige un sitio de noticias público (ej. [BioBioChile](https://www.biobiochile.cl/), [Emol](https://www.emol.com/), [CNN Chile](https://www.cnnchile.com/), [BBC Mundo](https://www.bbc.com/mundo)) que:
    - No requiera login.
    - Tenga estructura consistente (mismo layout para artículos).
    - Permita scraping ético (consulta `robots.txt` si es necesario).
- Usa Python con `requests` + `BeautifulSoup`, o bien con `selenium` si es un sitio dinámico.
- Extrae y almacena para **al menos 100 artículos**:
    - Título.
    - Fecha.
    - Cuerpo completo de la noticia.
    - URL original.
    - Categoría o sección (si es posible).
- Guarda los datos en un archivo `.json`, `.csv` o `.pkl` estructurado.
- Asegúrate de limpiar caracteres extraños, eliminar etiquetas HTML y normalizar el texto (acentos, espacios, etc).

In [2]:
###1. Configuración y utilidades

import re
import json
import unicodedata
from urllib.parse import urlparse
from datetime import datetime
import random

#Sitio objetivo (CNN Chile)
BASE = "https://www.cnnchile.com"
ROBOTS = f"{BASE}/robots.txt"
SITEMAP_INDEX = f"{BASE}/_files/sitemaps/sitemap_index.xml"
SITEMAP_NEWS  = f"{BASE}/_files/sitemaps/sitemap_news.xml"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; ProyScraper/1.0; +https://tu-dominio-ejemplo.cl/)",
    "Accept-Language": "es,es-CL;q=0.9,en;q=0.8"
}

CATEGORY_WHITELIST = {
    "pais","mundo","negocios","cultura","deportes","servicios","bits","miradas","opinion","cnn-data","futuro-360"
}

def fetch(url, retries=3, backoff=2.0, timeout=15):
    """DESCARGA ROBUSTA → devuelve BYTES (no texto)."""
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if 200 <= r.status_code < 300:
                return r.content  #clave para evitar mojibake
            time.sleep(backoff * (i + 1))
        except requests.RequestException:
            time.sleep(backoff * (i + 1))
    return None

def normalize_text(s: str) -> str:
    if not s:
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\xa0", " ").replace("\u200b", "")
    s = re.sub(r"\s+", " ", s, flags=re.M)
    return s.strip()

def is_article_url(url: str) -> bool:
    try:
        path = urlparse(url).path.strip("/")
        first = path.split("/", 1)[0].lower()
        if first in CATEGORY_WHITELIST:
            return True
        if first in {"tag","category","programas","en-vivo","search"}:
            return False
        return False
    except Exception:
        return False

In [3]:
###2. Sitemaps => recolectar URLs (>= 100)

def parse_xml_locs(xml_bytes) -> list:
    """Extrae <loc> desde XML (recibe BYTES)."""
    if not xml_bytes:
        return []
    soup = BeautifulSoup(xml_bytes, "xml")
    return [loc.get_text(strip=True) for loc in soup.find_all("loc")]

def get_sitemap_months():
    xml = fetch(SITEMAP_INDEX)
    months = parse_xml_locs(xml)
    months = [m for m in months if "/_files/sitemaps/" in m]
    months.sort(reverse=True)  #mas recientes primero
    return months

def get_news_urls(limit=200):
    urls = []

    news_xml = fetch(SITEMAP_NEWS)
    urls.extend(parse_xml_locs(news_xml))

    if len(urls) < limit:
        for sm in get_sitemap_months():
            xml = fetch(sm)
            urls.extend(parse_xml_locs(xml))
            if len(urls) >= limit * 2:
                break

    seen = set()
    filtered = []
    for u in urls:
        if not u or not u.startswith("http"):
            continue
        if is_article_url(u) and u not in seen:
            seen.add(u)
            filtered.append(u)

    filtered = [u for u in filtered if not re.search(r"\.(jpg|jpeg|png|webp)(\?|$)", u, re.I)]
    return filtered[:max(limit, 120)]

article_urls = get_news_urls(limit=200)
len(article_urls), article_urls[:5]

(173,
 ['https://www.cnnchile.com/mundo/javier-milei-debio-ser-evacuado-de-un-acto-electoral-tras-incidentes-con-manifestantes-opositores_20250827/',
  'https://www.cnnchile.com/pais/condena-gabriel-ruiz-tagle-informacion-privilegiada-mercado-valores-2025_20250827/',
  'https://www.cnnchile.com/deportes/10-hechos-de-agresividad-protagonizados-por-hinchas-de-independiente-y-que-desmienten-a-grindetti_20250827/',
  'https://www.cnnchile.com/pais/diputados-oposicion-delitos-violentos-fiestas-patrias_20250827/',
  'https://www.cnnchile.com/deportes/conmebol-define-fechas-cuartos-de-final-copa-libertadores_20250827/'])

In [4]:
###3. Parser de artículos (título, fecha, cuerpo, url, categoría)

DATE_PATTERNS = [
    ("%Y-%m-%dT%H:%M:%S%z", r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(?:Z|[+-]\d{2}:\d{2})"),
    ("%d.%m.%Y / %H:%M", r"\b\d{2}\.\d{2}\.\d{4}\s*/\s*\d{2}:\d{2}\b"),
]

BAD_PREFIXES = (
    "Lee también", "LEA TAMBIÉN", "Relacionado", "RELACIONADO", 
    "Video:", "VIDEO:", "Vea también", "VEA TAMBIÉN",
    "Compartir", "DESTACAMOS", "LO ÚLTIMO", "PROGRAMAS DE TV"
)

def parse_date_from_meta(soup: BeautifulSoup) -> str | None:
    metas = [
        ("meta", {"property": "article:published_time"}),
        ("meta", {"name": "article:published_time"}),
        ("meta", {"property": "og:updated_time"}),
        ("meta", {"name": "date"}),
        ("meta", {"itemprop": "datePublished"}),
        ("time",  {"itemprop": "datePublished"}),
    ]
    for tag_name, attrs in metas:
        tag = soup.find(tag_name, attrs=attrs)
        if tag:
            content = tag.get("content") or tag.get_text(strip=True)
            if content:
                iso = content.replace("Z", "+00:00")
                try:
                    return datetime.fromisoformat(iso).isoformat()
                except Exception:
                    pass
    return None

def parse_date_fallback(page_text: str) -> str | None:
    for fmt, regex in DATE_PATTERNS:
        m = re.search(regex, page_text)
        if m:
            date_str = m.group(0)
            try:
                dt = datetime.strptime(date_str, fmt)
                return dt.isoformat()
            except Exception:
                continue
    return None

def get_category_from_url(url: str) -> str:
    path = urlparse(url).path.strip("/")
    return (path.split("/", 1)[0] if path else "").lower()

def extract_main_paragraphs(soup: BeautifulSoup) -> list[str]:
    candidates = []
    for tag in soup.find_all(["article", "section", "div"]):
        ps = tag.find_all("p")
        if len(ps) >= 3:
            candidates.append((len(ps), tag))
    candidates.sort(key=lambda x: x[0], reverse=True)

    paragraphs = []
    if candidates:
        _, best = candidates[0]
        for p in best.find_all("p"):
            txt = normalize_text(p.get_text(" ", strip=True))
            if not txt:
                continue
            if any(txt.startswith(bad) for bad in BAD_PREFIXES):
                continue
            if len(txt) < 25 and re.search(r"^([A-ZÁÉÍÓÚÑ# ]{3,}|[A-ZÁÉÍÓÚÑ0-9 ]{3,})$", txt):
                continue
            paragraphs.append(txt)
    return paragraphs

def parse_article(url: str) -> dict | None:
    html_bytes = fetch(url)
    if not html_bytes:
        return None

    soup = BeautifulSoup(html_bytes, "html.parser")

    h1 = soup.find("h1")
    title = normalize_text(h1.get_text(strip=True)) if h1 else ""

    date_iso = parse_date_from_meta(soup)
    if not date_iso:
        page_text = soup.get_text("\n", strip=True)
        date_iso = parse_date_fallback(page_text)

    body_paragraphs = extract_main_paragraphs(soup)
    body = normalize_text(" ".join(body_paragraphs))

    categoria = get_category_from_url(url)

    #Fallback JSON-LD para completar título/cuerpo/fecha
    if not title or not body or len(body) < 200:
        for script in soup.find_all("script", type="application/ld+json"):
            try:
                data = json.loads(script.string or "")
                items = [data] if isinstance(data, dict) else (data if isinstance(data, list) else [])
                for item in items:
                    if isinstance(item, dict) and item.get("@type") in {"NewsArticle","Article"}:
                        title = title or normalize_text(item.get("headline",""))
                        body = body or normalize_text(item.get("articleBody",""))
                        if not date_iso and item.get("datePublished"):
                            try:
                                iso = item["datePublished"].replace("Z","+00:00")
                                date_iso = datetime.fromisoformat(iso).isoformat()
                            except Exception:
                                pass
            except Exception:
                continue

    if not title or not body or len(body) < 200:
        return None

    return {
        "titulo": title,
        "fecha": date_iso or "",
        "cuerpo": body,
        "url": url,
        "categoria": categoria,
    }

In [5]:
###4. Ejecutar scraping (>=100), validar y guardar

def polite_sleep():
    time.sleep(0.7 + random.random() * 0.5)

records = []
for i, url in enumerate(article_urls):
    art = parse_article(url)
    if art:
        records.append(art)
    if (i+1) % 20 == 0:
        print(f"Procesadas {i+1} URLs, válidas: {len(records)}")
    polite_sleep()
    if len(records) >= 130:  #sobrecolectar
        break

df = pd.DataFrame.from_records(records)
df = df.drop_duplicates(subset=["url"]).dropna(subset=["titulo","cuerpo"])
print(df.shape)
df.head(3)

Procesadas 20 URLs, válidas: 20
Procesadas 40 URLs, válidas: 40
Procesadas 60 URLs, válidas: 60
Procesadas 80 URLs, válidas: 80
Procesadas 100 URLs, válidas: 100
Procesadas 120 URLs, válidas: 120
(130, 5)


,titulo,fecha,cuerpo,url,categoria
0,Javier Milei debió ser evacuado de un acto ele...,2025-08-27T15:31:00,El presidente Javier Milei fue evacuado durant...,https://www.cnnchile.com/mundo/javier-milei-de...,mundo
1,Justicia condena a Gabriel Ruiz-Tagle por uso ...,2025-08-27T15:29:00,El exministro de Sebastián Piñera y expresiden...,https://www.cnnchile.com/pais/condena-gabriel-...,pais
2,“Acá no hay lugar para la violencia”: 8 hechos...,2025-08-27T15:29:00,Néstor Grindetti se ha encargado de responsabi...,https://www.cnnchile.com/deportes/10-hechos-de...,deportes


In [6]:
###5. Limpieza final y exportación

df["titulo"]   = df["titulo"].apply(normalize_text)
df["cuerpo"]   = df["cuerpo"].apply(normalize_text)
df["categoria"] = df["categoria"].str.lower().str.strip()

#Filtro de seguridad
df = df[df["cuerpo"].str.len() >= 400].copy()
df.reset_index(drop=True, inplace=True)

print("Total artículos listos:", len(df))
assert len(df) >= 100, "No se alcanzaron 100 artículos. Amplía el rango o relaja filtros."

CSV_OUT  = "cnnchile_noticias.csv"
JSON_OUT = "cnnchile_noticias.json"

#BOM para compatibilidad Excel/Windows
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")
df.to_json(JSON_OUT, orient="records", force_ascii=False)

CSV_OUT, JSON_OUT

Total artículos listos: 128


('cnnchile_noticias.csv', 'cnnchile_noticias.json')

## Parte 2: Diseño de Agentes Generadores de Contenido


### Instrucciones:
- Diseña **5 agentes con personalidades distintas**.
- Cada agente debe generar al menos 10 noticias basadas en el corpus extraído.
- Cada agente debe tener un comportamiento **automatizado y único** al generar contenido. Por ejemplo:

| Agente                 | Personalidad           | Comportamiento esperado                                                                                                                           |
| ---------------------- | ---------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------- |
| 1. Aleatorio           | "Publicador impulsivo" | Elige noticias al azar, publica un extracto sin cambios.                                                                                          |
| 2. Imitador            | "Papagayo"             | Copia noticias, pero cambia algunas palabras por sinónimos usando NLP básico.                                                                     |
| 3. Seleccionador       | "Curador temático"     | Filtra noticias de una temática y publica resúmenes cortos.                                                                                       |
| 4. Reescritor creativo | "Periodista junior"    | Reescribe noticias con un estilo personal (con reglas simples de cambio de redacción).                                                            |
| 5. Generador semántico | "Periodista LLM"       | Usa un modelo de lenguaje (`transformers`, `GPT`, `LLaMA`, etc.) para redactar nuevas versiones de las noticias originales, interpretándolas. |


In [7]:
# Define cada agente

agentes = [
    {
        "Agente": "Titulares Impactantes Contextuales",
        "Personalidad": "Cazador de clics con criterio",
        "Comportamiento esperado": (
            "Genera titulares clickbait ajustados a la categoría (economía, política, mundo, tech, etc.) "
            "usando bancos de frases por temática. Nunca sensacionaliza tragedias; en esos casos usa un titular neutro. "
            "Mantiene cifras/fechas intactas y añade un breve gancho de intriga. Produce ≥10 piezas por categoría."
        ),
    },
    {
        "Agente": "Cronista de Cifras",
        "Personalidad": "Periodista de datos",
        "Comportamiento esperado": (
            "Extrae números/porcentajes/fechas/montos y publica una nota 'Las cifras clave' con 3–6 bullets "
            "+ 1 oración final explicativa sin conjeturas. Aplica chequeos simples (p. ej., porcentajes ~100, duplicados). "
            "Genera ≥10 notas con artículos que tengan ≥2 magnitudes numéricas."
        ),
    },
    {
        "Agente": "Profe Claro",
        "Personalidad": "Explicador",
        "Comportamiento esperado": (
            "Reescribe en lenguaje sencillo: frases cortas, vocabulario común, sin jerga. "
            "Mantiene nombres y cifras. Estructura: título simplificado + 4–6 oraciones + mini glosario "
            "(3 términos técnicos definidos en 1 línea). Genera ≥10 piezas con tecnicismos."
        ),
    },
    {
        "Agente": "Vidente Tecnológico",
        "Personalidad": "Futurólogo pragmático",
        "Comportamiento esperado": (
            "Reescribe la noticia proyectándola a escenarios plausibles (p. ej., 2028/2030). "
            "1 párrafo de tendencia (qué pasa) + 1 de proyección (qué podría pasar) usando solo pistas del texto; "
            "no inventa hechos. Añade 'Riesgos y oportunidades' con 2 bullets. Genera ≥10 piezas en temas con tendencia clara."
        ),
    },
    {
        "Agente": "Especulador de Fuentes",
        "Personalidad": "Columnista opinante",
        "Comportamiento esperado": (
            "Produce un sumario de la nota y cierra con 1–2 opiniones editoriales en primera persona "
            "y una cita atribuida estilo análisis ('según analistas del sector…'), dejando claro que es opinión. "
            "No introduce datos nuevos; todo juicio se apoya en pasajes del artículo. Genera ≥10 columnas variadas."
        ),
    },
]

agentes_df = pd.DataFrame(agentes, columns=["Agente", "Personalidad", "Comportamiento esperado"])
agentes_df

,Agente,Personalidad,Comportamiento esperado
0,Titulares Impactantes Contextuales,Cazador de clics con criterio,Genera titulares clickbait ajustados a la cate...
1,Cronista de Cifras,Periodista de datos,Extrae números/porcentajes/fechas/montos y pub...
2,Profe Claro,Explicador,"Reescribe en lenguaje sencillo: frases cortas,..."
3,Vidente Tecnológico,Futurólogo pragmático,Reescribe la noticia proyectándola a escenario...
4,Especulador de Fuentes,Columnista opinante,Produce un sumario de la nota y cierra con 1–2...


In [14]:
import pandas as pd
import re
import random
from collections import defaultdict

# Cargar los datos
df = pd.read_csv('cnnchile_noticias.csv')

###1er agente de clickbait

# Definir plantillas de titulares por categoría
plantillas_clickbait = {
    "pais": [
        "¡Impactante! {} - Esto cambia todo",
        "Descubre la verdad sobre {} que nadie te había contado",
        "{}: Lo que las autoridades no quieren que sepas",
        "¿Sabías esto sobre {}? Te sorprenderá",
        "{}: El secreto mejor guardado revelado"
    ],
    "mundo": [
        "Alerta global: {} - Así afecta a Chile",
        "{}: La revelación que sacude al mundo",
        "Internacional: {} - Lo que realmente significa",
        "{}: La historia detrás de las noticias",
        "Mundo: {} - Una perspectiva única"
    ],
    "deportes": [
        "¡Increíble! {} - No vas a creer lo que pasó",
        "{}: El momento que cambió todo",
        "Deportes: {} - La jugada maestra revelada",
        "{}: Lo que nunca viste en televisión",
        "¡Insólito! {} - Así se vivió el momento clave"
    ],
    "cultura": [
        "{}: El detrás de cámaras que todos esperaban",
        "Cultura: {} - La revelación exclusiva",
        "{}: Lo que no sabías sobre este evento",
        "Arte y espectáculos: {} - Todo lo que debes saber",
        "{}: El secreto mejor guardado de la industria"
    ],
    "negocios": [
        "Economía: {} - Lo que significa para tu bolsillo",
        "{}: El dato que cambiará tus inversiones",
        "Finanzas: {} - La oportunidad que no puedes perder",
        "{}: Lo que los expertos no te dicen",
        "Negocios: {} - La estrategia revelada"
    ],
    "servicios": [
        "Importante: {} - Afecta directamente a tu familia",
        "{}: El cambio que necesitas conocer",
        "Servicios: {} - Lo que debes hacer ahora",
        "{}: La guía completa que te ayudará",
        "Actualización: {} - Todo lo nuevo que debes saber"
    ]
}

# Palabras clave que indican noticias trágicas (para evitar sensacionalismo)
palabras_tragedia = [
    "muere", "fallece", "fallecido", "fallecida", "muerte", "asesinato", 
    "tragedia", "accidente", "funeral", "velorio", "duelo", "luto",
    "desastre", "catástrofe", "víctima", "fatal", "fatalidad", "herido",
    "herida", "hospitalizado", "atentado", "ataque", "violencia", "sangre"
]

def es_noticia_tragica(titulo, cuerpo):
    """Determina si una noticia es trágica basándose en palabras clave"""
    texto_completo = (titulo + " " + cuerpo).lower()
    return any(palabra in texto_completo for palabra in palabras_tragedia)

def extraer_cifras_fechas(texto):
    """Extrae cifras y fechas del texto para preservarlas"""
    # Patrón para fechas (formats comunes)
    patron_fechas = r'\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b|\b\d{1,2}\s+(de\s+)?\w+\s+(de\s+)?\d{4}\b'
    # Patrón para cifras (números con símbolos de moneda, porcentajes, etc.)
    patron_cifras = r'\$\s*\d+[\d.,]*|\d+[\d.,]*\s*\%|\b\d+[\d.,]*(?:\.\d+)?\b'
    
    fechas = re.findall(patron_fechas, texto)
    cifras = re.findall(patron_cifras, texto)
    
    return fechas, cifras

def generar_titular_clickbait(titulo_original, cuerpo, categoria):
    """Genera un titular clickbait contextual apropiado para la categoría"""
    # Determinar si es noticia trágica
    if es_noticia_tragica(titulo_original, cuerpo):
        # Para noticias trágicas, usar un titular más neutro pero informativo
        return f"{titulo_original} - Información actualizada"
    
    # Extraer cifras y fechas importantes para preservarlas
    fechas, cifras = extraer_cifras_fechas(titulo_original)
    elementos_importantes = fechas + cifras
    
    # Seleccionar plantilla aleatoria según categoría
    if categoria not in plantillas_clickbait:
        categoria = "pais"  # Categoría por defecto
    
    plantilla = random.choice(plantillas_clickbait[categoria])
    
    # Acortar el titular original si es muy largo
    palabras = titulo_original.split()
    if len(palabras) > 8:
        titular_reducido = " ".join(palabras[:8]) + "..."
    else:
        titular_reducido = titulo_original
    
    # Aplicar plantilla
    nuevo_titular = plantilla.format(titular_reducido)
    
    # Asegurarse de incluir cifras y fechas importantes
    for elemento in elementos_importantes:
        if elemento not in nuevo_titular:
            nuevo_titular += f" | {elemento}"
    
    return nuevo_titular

In [15]:
# Ejecución de cada agente sobre el corpus

# Generar titulares para cada categoría
agente_noticias = []
titulares_por_categoria = defaultdict(list)

for _, row in df.iterrows():
    categoria = row['categoria']
    if categoria not in plantillas_clickbait:
        continue
        
    if len(titulares_por_categoria[categoria]) < 10:
        nuevo_titular = generar_titular_clickbait(
            row['titulo'], 
            row['cuerpo'], 
            categoria
        )
        
        titulares_por_categoria[categoria].append({
            'titulo_original': row['titulo'],
            'titulo_clickbait': nuevo_titular,
            'categoria': categoria,
            'url': row['url']
        })
        agente_noticias.append({
            'titulo_original': row['titulo'],
            'titulo_clickbait': nuevo_titular,
            'categoria': categoria,
            'url': row['url']
        })

# Asegurar al menos 10 titulares por categoría
for categoria in plantillas_clickbait:
    while len(titulares_por_categoria[categoria]) < 10:
        # Buscar noticias adicionales de esta categoría
        noticias_categoria = df[df['categoria'] == categoria]
        if len(noticias_categoria) > 0:
            for _, row in noticias_categoria.iterrows():
                if len(titulares_por_categoria[categoria]) < 10:
                    nuevo_titular = generar_titular_clickbait(
                        row['titulo'], 
                        row['cuerpo'], 
                        categoria
                    )
                    
                    titulares_por_categoria[categoria].append({
                        'titulo_original': row['titulo'],
                        'titulo_clickbait': nuevo_titular,
                        'categoria': categoria,
                        'url': row['url']
                    })
                    agente_noticias.append({
                        'titulo_original': row['titulo'],
                        'titulo_clickbait': nuevo_titular,
                        'categoria': categoria,
                        'url': row['url']
                    })
                else:
                    break

# Convertir a DataFrame y guardar resultados
df_agente = pd.DataFrame(agente_noticias)
df_agente.to_csv('titulares_clickbait_contextuales.csv', index=False, encoding='utf-8-sig')

# Mostrar algunos ejemplos
print("Ejemplos de titulares generados por el agente:")
print("=" * 80)
for i, row in df_agente.head(15).iterrows():
    print(f"Categoría: {row['categoria']}")
    print(f"Original: {row['titulo_original']}")
    print(f"Clickbait: {row['titulo_clickbait']}")
    print("-" * 80)

Ejemplos de titulares generados por el agente:
Categoría: mundo
Original: Javier Milei debió ser evacuado de un acto electoral tras incidentes con manifestantes opositores
Clickbait: Mundo: Javier Milei debió ser evacuado de un acto... - Una perspectiva única
--------------------------------------------------------------------------------
Categoría: pais
Original: Justicia condena a Gabriel Ruiz-Tagle por uso de información privilegiada: Lectura de sentencia quedó fijada
Clickbait: Descubre la verdad sobre Justicia condena a Gabriel Ruiz-Tagle por uso de... que nadie te había contado
--------------------------------------------------------------------------------
Categoría: deportes
Original: “Acá no hay lugar para la violencia”: 8 hechos de agresividad protagonizados por hinchas de Independiente y que desmienten a Grindetti
Clickbait: “Acá no hay lugar para la violencia”: 8 hechos de agresividad protagonizados por hinchas de Independiente y que desmienten a Grindetti - Información act

## Parte 3: Generación de portal de noticias en HTML


### Instrucciones:

- Crea un sitio web HTML básico (una sola página o varias secciones).
- Usa los textos generados por los agentes para crear un portal de noticias HTML.
- Muestra las noticias generadas por tus agentes, agrupadas por autor (agente).
- Incluye:
    - Título.
    - Fecha.
    - Nombre del agente.
    - Cuerpo del texto generado.
    - Imágenes (opcional).
- Puedes estilizar con CSS básico si lo deseas, pero se evalúa el contenido funcional.
- Puedes generar el HTML directamente desde Python con `Jinja2`, o usando templates simples.

In [ ]:
# Código para generar HTML con noticias agrupadas por agente

## Reflexión final


Crea un archivo `README.md` y responde:
- ¿Cuál agente generó contenido más coherente?
- ¿Qué dificultades tuviste al automatizar la generación de noticias?
- ¿Qué herramientas o métodos usarías si lo hicieras de nuevo?


## Entregables

Un archivo comprimido `.zip` con nombre `T1_TEL351_Nombre_Apellido.zip` que contenga:

1. **Jupyter Notebook** con todo el proceso:

    - Scraping

    - Procesamiento de datos

    - Implementación de los agentes

    - Generación del sitio

2. **Carpeta** `datos/` con el corpus original y el generado por los agentes.

3. **Carpeta** `sitio/` con archivos HTML y medios asociados.

4. **Archivo** `README.md`